In [1]:
import ir_datasets
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import nltk

dataset = ir_datasets.load("wikir/en1k/training")
doc_generator = (doc.text for doc in dataset.docs_iter())
print("docs generator created!")

nltk.download("punkt_tab")
nltk.download("wordnet")

doc_generator = (doc.text for doc in dataset.docs_iter())
lemmatized_docs = []
lematizer = WordNetLemmatizer()

for text in doc_generator:
    tokens = word_tokenize(text)
    lemmatized_words = [lematizer.lemmatize(word) for word in tokens]
    lemmatized_docs.append(" ".join(lemmatized_words))

print("The texts lematized!")

docs generator created!


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/tahas44/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /home/tahas44/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


The texts lematized!


In [2]:
from rank_bm25 import BM25Okapi

tokenized_corpus = [doc.split(" ") for doc in lemmatized_docs]
bm25 = BM25Okapi(tokenized_corpus)

In [3]:
queries = [query.text for query in dataset.queries_iter()]

lemmatized_queries = []

for query in queries:
    tokens = word_tokenize(query)
    lemmatized_words = [lematizer.lemmatize(word) for word in tokens]
    lemmatized_queries.append(" ".join(lemmatized_words))

scores_of_all_queries = []

for query in lemmatized_queries:
    scores_of_all_queries.append(bm25.get_scores(query.split(" ")))

In [4]:
scores_of_all_queries

[array([0., 0., 0., ..., 0., 0., 0.], shape=(369721,)),
 array([0., 0., 0., ..., 0., 0., 0.], shape=(369721,)),
 array([0., 0., 0., ..., 0., 0., 0.], shape=(369721,)),
 array([0., 0., 0., ..., 0., 0., 0.], shape=(369721,)),
 array([5.64339571, 8.6859124 , 5.35019992, ..., 5.87069785, 6.46225162,
        6.18193957], shape=(369721,)),
 array([ 0.        ,  0.        , 11.96163983, ...,  0.        ,
         0.        ,  0.        ], shape=(369721,)),
 array([0.        , 0.        , 0.        , ..., 0.44390768, 0.        ,
        0.        ], shape=(369721,)),
 array([0., 0., 0., ..., 0., 0., 0.], shape=(369721,)),
 array([0., 0., 0., ..., 0., 0., 0.], shape=(369721,)),
 array([5.87069785, 5.04563932, 6.39452708, ..., 6.29315245, 4.88659708,
        5.33363363], shape=(369721,)),
 array([5.01369053, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ], shape=(369721,)),
 array([0., 0., 0., ..., 0., 0., 0.], shape=(369721,)),
 array([0., 0., 0., ..., 0., 0., 0.], shap

In [5]:
from collections import defaultdict
from helper import Scoredoc

doc_dict = defaultdict(str)

for i, doc in enumerate(dataset.docs_iter()):
    doc_dict[i] = doc.doc_id

doc_dict = dict(doc_dict)

qrels_dict = defaultdict(list)

for qrel in dataset.qrels_iter():
    qrels_dict[qrel.query_id].append(qrel.doc_id)

qrels_dict = dict(qrels_dict)

score_doc_dict = defaultdict(list)

for scoreddoc in dataset.scoreddocs_iter():
    doc_id = scoreddoc.doc_id
    score = scoreddoc.score

    scoreddoc_object = Scoredoc(doc_id, score)

    score_doc_dict[scoreddoc.query_id].append(scoreddoc_object)

score_doc_dict = dict(score_doc_dict)

print("Necessary dicts created!")

Necessary dicts created!


In [6]:
import pandas as pd
query_ids = [query.query_id for query in dataset.queries_iter()]
df = pd.DataFrame(query_ids, columns=["Query_ID"])
df

,Query_ID
0,123839
1,188629
2,13898
3,316959
4,515031
...,...
1439,896124
1440,12319
1441,4421
1442,296526


In [7]:
from helper import create_AP, create_ndcg, create_statistical_columns, print_columns
df = create_statistical_columns(df, qrels_dict, doc_dict, scores_of_all_queries)
df = create_AP(df, qrels_dict, doc_dict, scores_of_all_queries)
df = create_ndcg(df, doc_dict, scores_of_all_queries, score_doc_dict)

print_columns(df)

recall_5_mean: 14.728414806575463
recall_5_std: 14.641829981888392
recall_5_max: 83.33333333333334
recall_5_min: 0.0
recall_10_mean: 20.849808309990394
recall_10_std: 19.549818005165868
recall_10_max: 100.0
recall_10_min: 0.0
precision_5_mean: 30.05540166204986
precision_5_std: 23.075932506196082
precision_5_max: 100.0
precision_5_min: 0.0
precision_10_mean: 22.451523545706372
precision_10_std: 17.208549315423763
precision_10_max: 100.0
precision_10_min: 0.0
f_score_5_mean: 18.39211574104246
f_score_5_std: 16.89189506569884
f_score_5_max: 90.9090909090909
f_score_5_min: 0.0
f_score_10_mean: 19.546604128172113
f_score_10_std: 16.537256965445074
f_score_10_max: 88.88888888888889
f_score_10_min: 0.0
MAP_5: 0.11650332900258151
MAP_10: 0.14287829859126994
NDCG_5_mean: 0.4949246124307086
NDCG_5_std: 0.3521514206794798
NDCG_5_max: 1.0000000000000002
NDCG_5_min: 0.0
NDCG_10_mean: 0.49531805498798337
NDCG_10_std: 0.3493172489447514
NDCG_10_max: 1.0000000000000002
NDCG_10_min: 0.0
